# Dish Up student survey: data cleaning

**Input:** `dishup_survey_anonymised.csv`, 282 survey responses (June–September 2026). Emails were removed before analysis.

**Output:** `dish_up_surveys_clean.csv`, 280 responses with tidy school groups and Tallinn districts, used by the analysis notebooks.

**Steps:** remove flagged rows → tidy text → group schools → map home areas to Tallinn's 8 districts → save.

In [1]:
import pandas as pd

## 1. Load and look

In [2]:
df = pd.read_csv("dishup_survey_anonymised.csv")
print(df.shape)
df.head()

(282, 8)


,response_id,submitted_at,school,home_area_raw,home_area,price_eur,launch_opt_in,flag
0,R001,2026-09-29 10:28:00,Estonian Academy of Arts,Kalamaja,Kalamaja,5,0,NaN
1,R002,2026-09-09 14:34:00,Tallinn University,Kesklinn,Kesklinn,6,1,NaN
2,R003,2026-09-07 16:13:00,Tallinn University,Kesklinn,Kesklinn,6,1,NaN
3,R004,2026-09-03 18:51:00,Tallinn University,Pirita,Pirita,5,1,NaN
4,R005,2026-09-03 18:33:00,Tallinn University,Aespa,Outside Tallinn,5,1,NaN


In [3]:
df.isna().sum()

response_id        0
submitted_at       0
school             0
home_area_raw      0
home_area          0
price_eur          0
launch_opt_in      0
flag             280
dtype: int64

## 2. Remove rows that shouldn't be analysed
Two rows are flagged: one person who answered twice (older answer kept out) and one internal test response.

In [4]:
df["flag"].value_counts()

flag
duplicate        1
internal_test    1
Name: count, dtype: int64

In [5]:
keep = df["flag"].isna()
clean = df[keep].copy()
print(len(df), "->", len(clean), "rows")

282 -> 280 rows


## 3. Tidy text

In [6]:
# Remove stray spaces at the start/end of text
for col in ["home_area_raw", "school"]:
    clean[col] = clean[col].str.strip()

## 4. Group schools
Three schools make up 92% of responses. Every other school has 1–9 responses, so they are grouped as "Other".

In [7]:
clean["school"].value_counts()

school
Tallinn University                                   183
TalTech                                               50
EBS                                                   25
High school student                                    9
University of Tartu                                    3
Estonian Academy of Arts                               1
Tartu University                                       1
Tallinna Tööstushariduskeskus                          1
Erasmus University of Rotterdam                        1
Tallinn Health University of Applied Sciences          1
TTHK                                                   1
MURG (mustamäe riigigümnaasium)                        1
Järvamaa rakenduslik kolledz                           1
Tallinna Majanduskool                                  1
University of Helsinki, but now I live in Tallinn      1
Name: count, dtype: int64

In [8]:
big_three = ["Tallinn University", "TalTech", "EBS"]

clean["school_group"] = clean["school"]
clean.loc[~clean["school"].isin(big_three), "school_group"] = "Other"

clean["school_group"].value_counts()

school_group
Tallinn University    183
TalTech                50
EBS                    25
Other                  22
Name: count, dtype: int64

## 5. Map home areas to Tallinn's 8 districts
Students typed their home area freely, so the same place appears with different spellings, as a neighbourhood, or outside Tallinn.

In [9]:
# Same capitalisation for everyone (e.g. viimsi -> Viimsi)
clean["area_title"] = clean["home_area_raw"].str.title()
clean["area_title"].value_counts()

area_title
Kesklinn                             54
Mustamäe                             41
Kristiine                            31
Põhja-Tallinn                        26
Lasnamäe                             20
Pirita                               15
Kadriorg                             14
Nõmme                                12
Haabersti                            12
Viimsi                               10
Kalamaja                              9
Õismäe                                4
Kakumäe                               4
Maardu                                2
Saku                                  2
Merivälja                             2
Männiku                               2
Tabasalu                              2
Aespa                                 1
Pääsküla/Nõmme                        1
10-15 Min Away From Tallinn           1
Vanalinn                              1
Keila                                 1
Kehra                                 1
Paldiski                     

In [10]:
districts = ["Kesklinn", "Kristiine", "Mustamäe", "Nõmme",
             "Haabersti", "Põhja-Tallinn", "Lasnamäe", "Pirita"]

# Neighbourhoods and typos -> their official district
fixes = {
    "Kadriorg": "Kesklinn",
    "Vanalinn": "Kesklinn",            # Old Town
    "Kalamaja": "Põhja-Tallinn",
    "Tondi": "Kristiine",
    "Õismäe": "Haabersti",
    "Väike-Õismäe": "Haabersti",
    "Haabersti (Väike-Õismäe)": "Haabersti",
    "Kakumäe": "Haabersti",
    "Haaberti": "Haabersti",           # typo
    "Merivälja": "Pirita",
    "Männiku": "Nõmme",
    "Pääsküla/Nõmme": "Nõmme",
}
clean["district"] = clean["area_title"].replace(fixes)

# Anything that isn't one of the 8 districts is outside Tallinn (incl. Viimsi) -> Other
clean.loc[~clean["district"].isin(districts), "district"] = "Other"

clean["district"].value_counts()

district
Kesklinn         69
Mustamäe         41
Põhja-Tallinn    35
Kristiine        32
Other            28
Haabersti        23
Lasnamäe         20
Pirita           17
Nõmme            15
Name: count, dtype: int64

In [11]:
# Check: what each raw answer was mapped to
clean[["home_area_raw", "district"]].drop_duplicates().sort_values("district")

,home_area_raw,district
18,Haabersti,Haabersti
239,Väike-Õismäe,Haabersti
72,Kakumäe,Haabersti
61,haabersti,Haabersti
9,Õismäe,Haabersti
155,Haabersti (Väike-Õismäe),Haabersti
191,Haaberti,Haabersti
1,Kesklinn,Kesklinn
5,Kadriorg,Kesklinn
35,Vanalinn,Kesklinn


## 6. Sanity checks

In [12]:
print("Prices:", clean["price_eur"].min(), "to", clean["price_eur"].max(), "EUR")
print("Opt-in values:", clean["launch_opt_in"].unique())
print("Dates:", clean["submitted_at"].min(), "to", clean["submitted_at"].max())

Prices: 4 to 10 EUR
Opt-in values: [0 1]
Dates: 2026-06-29 11:58:00 to 2026-09-29 10:28:00


## 7. Save
Keep the raw home area for traceability; drop helper columns.

In [13]:
clean = clean.drop(columns=["home_area", "area_title", "flag"])
clean.to_csv("dish_up_surveys_clean.csv", index=False)

print(clean.shape)
clean.head()

(280, 8)


,response_id,submitted_at,school,home_area_raw,price_eur,launch_opt_in,school_group,district
0,R001,2026-09-29 10:28:00,Estonian Academy of Arts,Kalamaja,5,0,Other,Põhja-Tallinn
1,R002,2026-09-09 14:34:00,Tallinn University,Kesklinn,6,1,Tallinn University,Kesklinn
2,R003,2026-09-07 16:13:00,Tallinn University,Kesklinn,6,1,Tallinn University,Kesklinn
3,R004,2026-09-03 18:51:00,Tallinn University,Pirita,5,1,Tallinn University,Pirita
4,R005,2026-09-03 18:33:00,Tallinn University,Aespa,5,1,Tallinn University,Other


**Result:** 280 responses · 4 school groups (Tallinn University 183, TalTech 50, EBS 25, Other 22) · 8 Tallinn districts + Other.

**Judgement calls:** Viimsi is a separate parish, so it counts as "Other" (outside Tallinn). Schools with fewer than 10 responses are grouped as "Other".